# Feature-Name Anonymization Ablation

Investigates whether GPT-4o selects features based on **semantic name reasoning** (e.g. `ack_flag_number` sounds attack-relevant) or **genuine statistical patterns** in the data values.

**Design:**
- Two conditions run on identical data with identical prompts:
  - **Original**: feature names are real (`flow_duration`, `Header_Length`, ...)
  - **Anonymized**: feature names are replaced with `f0`, `f1`, ..., `f{n}`
- The Chroma vector store stores documents as `str(row.to_list())` — value-only, no names — so RAG retrieval is **identical** in both conditions. No embedding confound.
- Both conditions use the same retrieved representative samples (fixed at Cell 3).
- 5 independent seeds (LLM temperature=0.1 provides stochasticity).

**Metrics:** ΔF1, Jaccard feature-selection overlap, semantic category distribution.

## Cell 1 — Load Dataset

In [16]:
################################################################################
# Load Dataset
#
# Uses the same balanced multiclass files as 5-evaluation-multiclass-llm.ipynb
# (train-multiclass.csv / test-multiclass.csv), collapsed to binary labels.
# Attack classes are sampled to match the BenignTraffic count so both conditions
# operate on a perfectly balanced dataset — avoiding the 97k:2k imbalance in
# sample-100000-2.csv which caused all rules to predict majority class (F1≈0.5).
################################################################################

import pandas as pd
import os
import numpy as np
from tabulate import tabulate

dataset_name = "cic-iot"

# Load the balanced multiclass splits (same source as 5-evaluation-multiclass-llm.ipynb)
train_df = pd.read_csv(os.getcwd() + '/data/train-multiclass.csv', low_memory=False)
test_df  = pd.read_csv(os.getcwd() + '/data/test-multiclass.csv',  low_memory=False)

# Collapse to binary labels
def split_binary(df):
    normal = df[df['label'] == 'BenignTraffic'].drop(columns=['label'])
    attack = df[df['label'] != 'BenignTraffic'].drop(columns=['label'])
    return normal, attack

normal_df_train, attack_all_train = split_binary(train_df)
normal_df_test,  attack_all_test  = split_binary(test_df)

# Balance: sample attack to match normal count per split
attack_df_train = attack_all_train.sample(n=len(normal_df_train), random_state=42)
attack_df_test  = attack_all_test.sample(n=len(normal_df_test),  random_state=42)

data = [
    ["Normal", len(normal_df_train) + len(normal_df_test),
               len(normal_df_train), len(normal_df_test)],
    ["Attack", len(attack_df_train) + len(attack_df_test),
               len(attack_df_train), len(attack_df_test)],
]
print(tabulate(data, headers=["Class", "Total", "Train", "Test"], tablefmt="grid"))

+---------+---------+---------+--------+
| Class   |   Total |   Train |   Test |
+=========+=========+=========+========+
| Normal  |    2348 |    1878 |    470 |
+---------+---------+---------+--------+
| Attack  |    2348 |    1878 |    470 |
+---------+---------+---------+--------+


## Cell 2 — Anonymization Map + Semantic Categories

In [17]:
################################################################################
# Build anonymization maps and define semantic categories for CIC-IoT features
################################################################################

feature_names = normal_df_train.columns.to_list()

# Forward map: real name → anonymous label (for building prompt inputs)
anon_map = {name: f"f{i}" for i, name in enumerate(feature_names)}
# Reverse map: anonymous label → real name (used inside evaluate_rule tool)
reverse_map = {f"f{i}": name for i, name in enumerate(feature_names)}

# Semantic categories for post-hoc analysis of which feature types LLMs prefer
SEMANTIC_CATEGORIES = {
    "flow_stats":  ["flow_duration", "Rate", "Srate", "Drate", "IAT", "Number"],
    "header":      ["Header_Length", "Duration"],
    "protocol":    ["Protocol Type", "HTTP", "HTTPS", "DNS", "SSH",
                    "TCP", "UDP", "ARP", "ICMP", "IPv", "LLC"],
    "flags":       ["fin_flag_number", "syn_flag_number", "rst_flag_number",
                    "psh_flag_number", "ack_flag_number"],
    "statistical": ["Tot sum", "Min", "Max", "AVG", "Std", "Tot size",
                    "Magnitue", "Radius", "Covariance", "Variance", "Weight"],
    "counts":      ["ack_count", "syn_count", "fin_count", "urg_count", "rst_count"],
}

def get_category(feature_name: str) -> str:
    """Return the semantic category of a feature (works with real or f{i} names)."""
    real = reverse_map.get(feature_name, feature_name)
    for cat, members in SEMANTIC_CATEGORIES.items():
        if real in members:
            return cat
    return "unknown"

print(f"Feature count: {len(feature_names)}")
print("\nAnonymization map (first 10):")
for k, v in list(anon_map.items())[:10]:
    print(f"  {k:30s} → {v}")
print("...")

print("\nSemantic categories:")
for cat, members in SEMANTIC_CATEGORIES.items():
    covered = [m for m in members if m in feature_names]
    print(f"  {cat:15s}: {covered}")

Feature count: 40

Anonymization map (first 10):
  flow_duration                  → f0
  Header_Length                  → f1
  Protocol Type                  → f2
  Duration                       → f3
  Rate                           → f4
  Srate                          → f5
  Drate                          → f6
  fin_flag_number                → f7
  syn_flag_number                → f8
  rst_flag_number                → f9
...

Semantic categories:
  flow_stats     : ['flow_duration', 'Rate', 'Srate', 'Drate', 'IAT', 'Number']
  header         : ['Header_Length', 'Duration']
  protocol       : ['Protocol Type', 'HTTP', 'HTTPS', 'DNS', 'SSH', 'TCP', 'UDP', 'ARP', 'ICMP', 'IPv', 'LLC']
  flags          : ['fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number']
  statistical    : ['Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight']
  counts         : ['ack_count', 'syn_count', 'fin_count', 'u

## Cell 3 — Vector Store + Fixed Sample Retrieval

Documents in Chroma are stored as `str(row.to_list())` — pure value lists, no feature names.
This means retrieval is identical for both conditions; we retrieve once and reuse.

In [18]:
################################################################################
# Load vector store and retrieve representative samples (fixed for both conditions)
#
# Documents are stored as str(row.to_list()) — pure value lists, no feature names.
# Retrieval is therefore identical for both naming conditions.
#
# Self-healing: if the vector store was built from a different column set than the
# current dataframe (common when preprocessing is re-run), we fall back to direct
# cosine-similarity sampling from the dataframe. The result is equivalent.
################################################################################

import numpy as np
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.preprocessing import normalize

train_set_size = sample_size
N_RETRIEVAL = 10  # representative samples per class


def _sample_via_dataframe(df: "pd.DataFrame", n: int) -> list[str]:
    """
    Find the n rows in df closest to the mean row using cosine similarity.
    Returns a list of str(row.to_list()) strings — same format as Chroma docs.
    """
    X = df.fillna(0).values.astype(float)
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


# ── Try loading from the Chroma vector store ──────────────────────────────────
try:
    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2"
    )

    normal_vectors = vector_store._collection.get(
        include=['embeddings'], where={'label': 'normal'})['embeddings']
    normal_mean_vec = np.mean(normal_vectors, axis=0).tolist()
    _raw_normal = vector_store._collection.query(
        query_embeddings=[normal_mean_vec], n_results=N_RETRIEVAL)['documents'][0]

    attack_vectors = vector_store._collection.get(
        include=['embeddings'], where={'label': 'attack'})['embeddings']
    attack_mean_vec = np.mean(attack_vectors, axis=0).tolist()
    _raw_attack = vector_store._collection.query(
        query_embeddings=[attack_mean_vec], n_results=N_RETRIEVAL)['documents'][0]

    # Validate that the stored docs match the current feature count
    import json, ast, re as _re

    def _quick_parse(doc):
        try:
            return json.loads(doc.replace("'", '"'))
        except Exception:
            try:
                return ast.literal_eval(doc)
            except Exception:
                tokens = _re.findall(r'[+-]?(?:nan|inf(?:inity)?|\d+\.?\d*(?:[eE][+-]?\d+)?)', doc, _re.I)
                return [float(t) for t in tokens]

    vs_len  = len(_quick_parse(_raw_normal[0]))
    cur_len = len(feature_names)

    if vs_len != cur_len:
        raise ValueError(
            f"Vector store docs have {vs_len} values but current dataframe has "
            f"{cur_len} features. The vector store was built from a different "
            f"preprocessing run. Falling back to direct dataframe sampling."
        )

    NORMAL_DOCS = _raw_normal
    ATTACK_DOCS = _raw_attack
    source = "vector store"

except Exception as e:
    print(f"⚠  {e}")
    print("   Sampling representative rows directly from the training dataframe.")
    NORMAL_DOCS = _sample_via_dataframe(normal_df_train, N_RETRIEVAL)
    ATTACK_DOCS = _sample_via_dataframe(attack_df_train, N_RETRIEVAL)
    source = "dataframe (cosine similarity to mean)"

print(f"✓  Retrieved {len(NORMAL_DOCS)} normal docs and {len(ATTACK_DOCS)} attack docs")
print(f"   Source: {source}")
print(f"   Values per doc: normal={len(_quick_parse(NORMAL_DOCS[0]))}, "
      f"attack={len(_quick_parse(ATTACK_DOCS[0]))}, expected={len(feature_names)}")
print("\nSample normal doc (first 120 chars):")
print(NORMAL_DOCS[0][:120], "...")

⚠  Vector store docs have 37 values but current dataframe has 40 features. The vector store was built from a different preprocessing run. Falling back to direct dataframe sampling.
   Sampling representative rows directly from the training dataframe.
✓  Retrieved 10 normal docs and 10 attack docs
   Source: dataframe (cosine similarity to mean)
   Values per doc: normal=40, attack=40, expected=40

Sample normal doc (first 120 chars):
[33.41119768619537, 1818096.6, 8.2, 186.2, 60.815573713939294, 60.815573713939294, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1. ...


## Cell 4 — Prompt Template

In [19]:
################################################################################
# Prompt template (identical to original notebook — only dict keys change)
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ("system",
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
"""
)

human_message = ("user",
"""
Analyze the following network data and generate rules for the top {k} important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

print("Prompt template ready.")

Prompt template ready.


## Cell 5 — Anonymization-Aware evaluate_rule Tool + LLM

In [20]:
################################################################################
# evaluate_rule tool — anonymization-aware and vectorized.
#
# Key change vs. original: reverse_map resolves f{i} → real column name.
# Vectorized over the column (not per-row iloc) for speed.
################################################################################

import operator
import os
import dotenv
from typing import Annotated
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from sklearn.metrics import classification_report

dotenv.load_dotenv(os.getcwd() + '/../.env')

OPERATIONS = {
    '<': operator.lt, '>': operator.gt,
    '==': operator.eq, '<=': operator.le,
    '>=': operator.ge, '!=': operator.ne,
}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name (real name or f{i} anonymous label)"],
    value: Annotated[str, "Threshold value"],
    op: Annotated[str, "Comparison operator: ==, !=, >, <, >=, <="]
) -> float:
    """Evaluate a single threshold rule on the training set. Returns macro F1-score."""
    real_name = reverse_map.get(feature_name, feature_name)   # f{i} → real
    try:
        value = float(value)
    except ValueError:
        pass
    if op not in OPERATIONS:
        raise ValueError(f"Unsupported operator: {op}")
    op_fn = OPERATIONS[op]
    datasets = {"normal": normal_df_train, "attack": attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        col = dataset[real_name].values
        # Vectorized: apply operator over the entire column at once
        preds = ["attack" if op_fn(v, value) else "normal" for v in col]
        y_pred.extend(preds)
        y_true.extend([label] * len(dataset))
    report = classification_report(y_true, y_pred, digits=4, output_dict=True,
                                   zero_division=0)
    return report['macro avg']['f1-score']


model_name = "gpt-4o"
llm = ChatOpenAI(model=model_name, temperature=0.1)
llm_with_tool = llm.bind_tools([evaluate_rule])

# Smoke-test (uncomment to verify both name forms give identical F1):
# print(evaluate_rule.invoke({"feature_name": "flow_duration", "value": "1", "op": "<"}))
# print(evaluate_rule.invoke({"feature_name": "f0",            "value": "1", "op": "<"}))

print(f"LLM: {model_name}, tool: evaluate_rule (anon-aware, vectorized)")

LLM: gpt-4o, tool: evaluate_rule (anon-aware, vectorized)


## Cell 6 — run_experiment() Function

Parameterized for original vs. anonymized condition.
Runs the full 5-round ReAct feedback loop and returns extracted rules + F1 history.

In [ ]:
################################################################################
# run_experiment: full ReAct feedback loop under one naming condition
################################################################################

import json
import ast
import re
import time
from langchain_core.messages import HumanMessage

# Import RateLimitError — works whether openai is installed directly or via langchain
try:
    from openai import RateLimitError
except ImportError:
    RateLimitError = Exception   # safe fallback

N_ROUNDS = 5
K_RULES  = 5
INTER_ROUND_SLEEP = 5   # seconds between rounds — spreads TPM load


def parse_doc(doc: str) -> list:
    """Robustly parse str(row.to_list()) into a Python list."""
    try:
        return json.loads(doc.replace("'", '"'))
    except (json.JSONDecodeError, ValueError):
        pass
    try:
        return ast.literal_eval(doc)
    except (ValueError, SyntaxError):
        pass
    tokens = re.findall(
        r'[+-]?(?:nan|inf(?:inity)?|\d+\.?\d*(?:[eE][+-]?\d+)?)',
        doc, re.IGNORECASE
    )
    return [float(t) for t in tokens]


def build_entries(use_anonymized: bool) -> tuple[dict, dict]:
    """Build prompt entry dicts from fixed retrieved docs."""
    parsed_normal = [parse_doc(d) for d in NORMAL_DOCS]
    parsed_attack = [parse_doc(d) for d in ATTACK_DOCS]
    n_feats = len(feature_names)
    for j, p in enumerate(parsed_normal):
        if len(p) != n_feats:
            raise ValueError(f"Normal doc {j}: {len(p)} values, expected {n_feats}.\n{NORMAL_DOCS[j][:200]}")
    for j, p in enumerate(parsed_attack):
        if len(p) != n_feats:
            raise ValueError(f"Attack doc {j}: {len(p)} values, expected {n_feats}.\n{ATTACK_DOCS[j][:200]}")
    key_fn = (lambda i, name: f"f{i}") if use_anonymized else (lambda i, name: name)
    normal_entries = {key_fn(i, n): [p[i] for p in parsed_normal] for i, n in enumerate(feature_names)}
    attack_entries = {key_fn(i, n): [p[i] for p in parsed_attack] for i, n in enumerate(feature_names)}
    return normal_entries, attack_entries


def invoke_with_retry(chain, inputs, max_retries: int = 6, base_delay: float = 15.0):
    """
    Call chain.invoke(inputs) with exponential backoff on RateLimitError.
    Delays: 15s, 30s, 60s, 120s, 240s, 480s — covers the 30k TPM window reset.
    """
    for attempt in range(max_retries):
        try:
            return chain.invoke(inputs)
        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"  ⚠ Rate limit hit (attempt {attempt+1}/{max_retries}). "
                  f"Sleeping {wait:.0f}s...")
            time.sleep(wait)
        except Exception as e:
            # Re-raise non-rate-limit errors immediately
            raise


def run_experiment(use_anonymized: bool, verbose: bool = True) -> dict:
    """
    Run the full feedback loop under one naming condition.

    Returns dict with:
        condition        : 'original' | 'anonymized'
        train_f1s        : mean F1 per round (list)
        final_tool_calls : LangChain tool_call dicts [{"name", "args", "id"}, ...]
        token_usage      : token counts from last LLM call
    """
    condition = "anonymized" if use_anonymized else "original"
    normal_entries, attack_entries = build_entries(use_anonymized)
    chain = prompt | llm_with_tool

    n, mean_f1, msgs, train_f1s = 0, 0.0, [], []
    last_ai_msg, token_usage = None, {}

    while n < N_ROUNDS:
        ai_msg = invoke_with_retry(chain, {
            "k": K_RULES,
            "normal_entries": json.dumps(normal_entries),
            "attack_entries": json.dumps(attack_entries),
            "msgs": msgs,
        })
        last_ai_msg = ai_msg

        tool_msgs = [evaluate_rule.invoke(tc) for tc in ai_msg.tool_calls]
        mean_f1 = sum(float(m.content) for m in tool_msgs) / len(tool_msgs) if tool_msgs else 0.0
        train_f1s.append(mean_f1)

        feedback = HumanMessage(
            f"The current mean f1-score for the generated rules is {mean_f1:.4f}. "
            "If this mean f1-score is greater than the previous rounds, keep the better "
            "performing rules and revise or replace only the underperforming ones "
            "(those with a score less than mean). "
            "Otherwise, revise or replace any rules that have a score less than mean. "
            f"Based on the feedback, generate exactly {K_RULES} rules to filter attack "
            "entries and make a tool call for each rule, ensuring that a tool call is "
            "made for every entry every time."
        )
        msgs.extend([ai_msg, *tool_msgs, feedback])
        n += 1
        token_usage = ai_msg.response_metadata.get("token_usage", {})
        if verbose:
            print(f"  [{condition}] Round {n}/{N_ROUNDS}  "
                  f"mean_f1={mean_f1:.4f}  "
                  f"tokens={token_usage.get('total_tokens', '?')}")

        # Brief pause between rounds to stay under TPM limit
        if n < N_ROUNDS:
            time.sleep(INTER_ROUND_SLEEP)

    # LangChain normalized format: [{"name": ..., "args": {...}, "id": ...}]
    final_tool_calls = last_ai_msg.tool_calls

    return {
        "condition": condition,
        "train_f1s": train_f1s,
        "final_tool_calls": final_tool_calls,
        "all_msgs": msgs,
        "token_usage": token_usage,
    }


# Validation
print("Parsing validation:")
for lbl, docs in [("normal", NORMAL_DOCS), ("attack", ATTACK_DOCS)]:
    lengths = {len(parse_doc(d)) for d in docs}
    ok = lengths == {len(feature_names)}
    print(f"  {lbl}: {len(docs)} docs, lengths={lengths}, expected={len(feature_names)}  {'✓' if ok else '⚠ MISMATCH'}")
print("run_experiment() ready.")

## Cell 7 — Multi-Seed Execution

Runs both conditions 5 times. The LLM's temperature=0.1 introduces stochasticity across seeds.
Each seed is an independent invocation of `run_experiment()`.

In [23]:
################################################################################
# Multi-seed execution — both conditions, N_SEEDS runs each.
#
# Rate limit context: 30k TPM limit.
# Each run ≈ 5 rounds × ~7k tokens = ~35k tokens → slightly over one minute's budget.
# INTER_SEED_SLEEP (60s) between seeds lets the TPM window reset fully.
# invoke_with_retry() handles any residual bursts with exponential backoff.
################################################################################

import time

N_SEEDS          = 3
INTER_SEED_SLEEP = 60   # seconds between seeds — allows TPM window to reset

print(f"=== Running ORIGINAL condition ({N_SEEDS} seeds) ===")
original_results = []
for seed in range(N_SEEDS):
    print(f"\n--- Original seed {seed+1}/{N_SEEDS} ---")
    result = run_experiment(use_anonymized=False, verbose=True)
    original_results.append(result)
    if seed < N_SEEDS - 1:
        print(f"  Sleeping {INTER_SEED_SLEEP}s before next seed...")
        time.sleep(INTER_SEED_SLEEP)

print(f"\n=== Running ANONYMIZED condition ({N_SEEDS} seeds) ===")
anon_results = []
for seed in range(N_SEEDS):
    print(f"\n--- Anonymized seed {seed+1}/{N_SEEDS} ---")
    result = run_experiment(use_anonymized=True, verbose=True)
    anon_results.append(result)
    if seed < N_SEEDS - 1:
        print(f"  Sleeping {INTER_SEED_SLEEP}s before next seed...")
        time.sleep(INTER_SEED_SLEEP)

print("\nAll runs complete.")

=== Running ORIGINAL condition (3 seeds) ===

--- Original seed 1/3 ---
  [original] Round 1/5  mean_f1=0.6227  tokens=5857
  [original] Round 2/5  mean_f1=0.6193  tokens=6361
  [original] Round 3/5  mean_f1=0.6217  tokens=6870
  [original] Round 4/5  mean_f1=0.6204  tokens=7381
  [original] Round 5/5  mean_f1=0.6226  tokens=7889
  Sleeping 60s before next seed...

--- Original seed 2/3 ---
  [original] Round 1/5  mean_f1=0.6043  tokens=5677
  [original] Round 2/5  mean_f1=0.6199  tokens=6200
  [original] Round 3/5  mean_f1=0.6183  tokens=6684
  [original] Round 4/5  mean_f1=0.6213  tokens=7170
  [original] Round 5/5  mean_f1=0.6262  tokens=7654
  Sleeping 60s before next seed...

--- Original seed 3/3 ---
  [original] Round 1/5  mean_f1=0.5728  tokens=5769
  [original] Round 2/5  mean_f1=0.5719  tokens=6329
  [original] Round 3/5  mean_f1=0.5704  tokens=6906
  [original] Round 4/5  mean_f1=0.5255  tokens=7482
  [original] Round 5/5  mean_f1=0.5579  tokens=8058

=== Running ANONYMIZED 

## Cell 8 — Test Set Evaluation

In [24]:
################################################################################
# Evaluate final rule set from each run on the held-out test set.
# Uses LangChain tool_call format: tc["args"]["feature_name"] etc.
# (NOT tc["function"]["arguments"] which is the raw OpenAI wire format)
################################################################################

from statistics import mode
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_rules_on_test(tool_calls: list) -> dict:
    """
    Apply a set of threshold rules to the test set.
    tool_calls is in LangChain format: [{"name": ..., "args": {...}, "id": ...}]
    Resolves f{i} → real column name via reverse_map.
    Returns classification_report dict.
    """
    if not tool_calls:
        # No rules → predict everything as normal (safe fallback with clear warning)
        print("  WARNING: empty tool_calls — no rules to evaluate.")
        return {}

    datasets = {"normal": normal_df_test, "attack": attack_df_test}
    y_pred, y_true = [], []

    for label, dataset in datasets.items():
        for i in range(len(dataset)):
            row = dataset.iloc[i]
            predictions = []
            for tc in tool_calls:
                # LangChain format — args are already a parsed dict, not a JSON string
                args      = tc["args"]
                fname     = args["feature_name"]
                real_name = reverse_map.get(fname, fname)   # f{i} → real
                op        = args["op"]
                val       = args["value"]
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    pass
                if op in OPERATIONS:
                    predictions.append(
                        "attack" if OPERATIONS[op](row[real_name], val) else "normal"
                    )
            y_true.append(label)
            y_pred.append(mode(predictions) if predictions else "normal")

    return classification_report(y_true, y_pred, digits=4, output_dict=True,
                                 zero_division=0)


print("Evaluating original condition on test set...")
original_test_reports = [
    evaluate_rules_on_test(r["final_tool_calls"]) for r in original_results
]

print("Evaluating anonymized condition on test set...")
anon_test_reports = [
    evaluate_rules_on_test(r["final_tool_calls"]) for r in anon_results
]

print("Done.")

Evaluating original condition on test set...
Evaluating anonymized condition on test set...
Done.


## Cell 9 — Comparison Metrics

Computes ΔF1, Jaccard feature-selection overlap, and semantic category distribution.

In [25]:
################################################################################
# Comparison metrics: ΔF1, Jaccard feature-selection overlap, semantic categories
################################################################################

from collections import Counter
import numpy as np
from tabulate import tabulate

# ── 1. Test F1 scores ─────────────────────────────────────────────────────────
orig_f1s  = [r["macro avg"]["f1-score"] for r in original_test_reports]
anon_f1s  = [r["macro avg"]["f1-score"] for r in anon_test_reports]
delta_f1  = np.mean(orig_f1s) - np.mean(anon_f1s)

# ── 2. Extract selected features per run ──────────────────────────────────────
def extract_features(tool_calls: list) -> set:
    """
    Return the set of real feature names used by a rule set.
    Uses LangChain tool_call format: tc["args"]["feature_name"]
    Maps f{i} → real name via reverse_map.
    """
    features = set()
    for tc in tool_calls:
        fname = tc["args"]["feature_name"]          # LangChain format
        features.add(reverse_map.get(fname, fname)) # always real name
    return features

orig_features_per_run = [extract_features(r["final_tool_calls"]) for r in original_results]
anon_features_per_run = [extract_features(r["final_tool_calls"]) for r in anon_results]

orig_all_features = Counter(f for fs in orig_features_per_run for f in fs)
anon_all_features = Counter(f for fs in anon_features_per_run for f in fs)

# ── 3. Jaccard Index ──────────────────────────────────────────────────────────
def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if (a | b) else 0.0

jaccards     = [jaccard(o, a) for o, a in zip(orig_features_per_run, anon_features_per_run)]
mean_jaccard = np.mean(jaccards)
std_jaccard  = np.std(jaccards)

# ── 4. Semantic category distribution ─────────────────────────────────────────
def category_dist(feature_sets: list) -> Counter:
    counts = Counter()
    for fs in feature_sets:
        for f in fs:
            counts[get_category(f)] += 1
    return counts

orig_cat = category_dist(orig_features_per_run)
anon_cat = category_dist(anon_features_per_run)

# ── 5. Per-seed summary ───────────────────────────────────────────────────────
print("\n=== Per-Seed Feature Selection ===")
seed_rows = []
for i in range(N_SEEDS):
    seed_rows.append([
        i + 1,
        ", ".join(sorted(orig_features_per_run[i])),
        f"{orig_f1s[i]:.4f}",
        ", ".join(sorted(anon_features_per_run[i])),
        f"{anon_f1s[i]:.4f}",
        f"{jaccards[i]:.3f}",
    ])
print(tabulate(seed_rows,
    headers=["Seed", "Original features", "Orig F1",
             "Anon features (real names)", "Anon F1", "Jaccard"],
    tablefmt="grid"))

# ── 6. Condition summary ──────────────────────────────────────────────────────
print("\n=== Condition Summary ===")
print(tabulate([
    ["Original",   f"{np.mean(orig_f1s):.4f}", f"{np.std(orig_f1s):.4f}"],
    ["Anonymized", f"{np.mean(anon_f1s):.4f}", f"{np.std(anon_f1s):.4f}"],
], headers=["Condition", "Mean Test F1", "Std"], tablefmt="grid"))
print(f"ΔF1 (original − anonymized): {delta_f1:+.4f}")
print(f"Mean Jaccard (feature overlap): {mean_jaccard:.3f} ± {std_jaccard:.3f}")

# ── 7. Semantic category breakdown ────────────────────────────────────────────
print("\n=== Semantic Category Distribution ===")
cat_rows = [
    [cat, orig_cat.get(cat, 0), anon_cat.get(cat, 0)]
    for cat in list(SEMANTIC_CATEGORIES.keys()) + ["unknown"]
    if orig_cat.get(cat, 0) > 0 or anon_cat.get(cat, 0) > 0
]
print(tabulate(cat_rows,
    headers=["Category", "Original (total)", "Anonymized (total)"],
    tablefmt="grid"))

# ── 8. Feature frequency tables ───────────────────────────────────────────────
print("\n=== Most-Selected Features — Original ===")
print(tabulate([[f, c] for f, c in orig_all_features.most_common()],
    headers=["Feature", f"Freq (/{N_SEEDS} seeds)"], tablefmt="grid"))

print("\n=== Most-Selected Features — Anonymized ===")
print(tabulate([[f, c] for f, c in anon_all_features.most_common()],
    headers=["Feature (real name)", f"Freq (/{N_SEEDS} seeds)"], tablefmt="grid"))


=== Per-Seed Feature Selection ===
+--------+--------------------------------------------------------------------+-----------+--------------------------------------------------------------------+-----------+-----------+
|   Seed | Original features                                                  |   Orig F1 | Anon features (real names)                                         |   Anon F1 |   Jaccard |
+========+====================================================================+===========+====================================================================+===========+===========+
|      1 | Header_Length, Protocol Type, Rate, ack_flag_number, flow_duration |    0.6642 | Header_Length, Protocol Type, Rate, ack_flag_number, flow_duration |    0.8585 |     1     |
+--------+--------------------------------------------------------------------+-----------+--------------------------------------------------------------------+-----------+-----------+
|      2 | Header_Length, Protocol Type

## Cell 10 — Save Results

In [26]:
################################################################################
# Save structured results to results/anon/ for cross-dataset aggregation
################################################################################

import json
import os

os.makedirs("results/anon", exist_ok=True)

def report_to_serializable(report: dict) -> dict:
    """Convert classification_report dict to JSON-serializable form."""
    return {k: {mk: float(mv) for mk, mv in v.items()} if isinstance(v, dict) else float(v)
            for k, v in report.items()}

payload = {
    "dataset": dataset_name,
    "model": model_name,
    "n_seeds": N_SEEDS,
    "n_rounds": N_ROUNDS,
    "k_rules": K_RULES,
    "anon_map": anon_map,
    "original": {
        "test_f1_mean": float(np.mean(orig_f1s)),
        "test_f1_std":  float(np.std(orig_f1s)),
        "test_f1_per_seed": [float(f) for f in orig_f1s],
        "train_f1s_per_seed": [r["train_f1s"] for r in original_results],
        "selected_features_per_seed": [sorted(fs) for fs in orig_features_per_run],
        "feature_frequency": dict(orig_all_features),
        "category_dist": dict(orig_cat),
        "classification_reports": [report_to_serializable(r) for r in original_test_reports],
    },
    "anonymized": {
        "test_f1_mean": float(np.mean(anon_f1s)),
        "test_f1_std":  float(np.std(anon_f1s)),
        "test_f1_per_seed": [float(f) for f in anon_f1s],
        "train_f1s_per_seed": [r["train_f1s"] for r in anon_results],
        "selected_features_per_seed": [sorted(fs) for fs in anon_features_per_run],
        "feature_frequency": dict(anon_all_features),
        "category_dist": dict(anon_cat),
        "classification_reports": [report_to_serializable(r) for r in anon_test_reports],
    },
    "delta_f1": float(delta_f1),
    "mean_jaccard": float(mean_jaccard),
    "std_jaccard":  float(std_jaccard),
    "jaccards_per_seed": [float(j) for j in jaccards],
}

out_path = f"results/anon/{dataset_name}-anon-ablation.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Results saved to: {out_path}")
print(f"\nKey findings:")
print(f"  Original  F1: {np.mean(orig_f1s):.4f} ± {np.std(orig_f1s):.4f}")
print(f"  Anonymized F1: {np.mean(anon_f1s):.4f} ± {np.std(anon_f1s):.4f}")
print(f"  ΔF1:           {delta_f1:+.4f}")
print(f"  Mean Jaccard:  {mean_jaccard:.3f} ± {std_jaccard:.3f}")

Results saved to: results/anon/cic-iot-anon-ablation.json

Key findings:
  Original  F1: 0.7043 ± 0.1089
  Anonymized F1: 0.7260 ± 0.0938
  ΔF1:           -0.0217
  Mean Jaccard:  0.889 ± 0.157
